# Wagner and Whitin's Dynamic Economic Lot-Size Model

Wagner and Whitin's finite-horizon lot-sizing problem chooses when and how much to order when demand and setup costs vary by period. Replenishment is instantaneous, shortages are forbidden, and initial and final inventory are zero. Because unit purchase cost is constant, only setup and inventory-holding costs affect the policy.

For order quantities $x_t$, ending inventories $I_t$, and setup indicators $y_t$, the objective is

$$\min \sum_{t=1}^{N}(s_ty_t+h_tI_t),$$

with inventory balance $I_{t-1}+x_t-I_t=d_t$. The paper's twelve-month example uses:

| Month | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 | 11 | 12 |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| Demand $d_t$ | 69 | 29 | 36 | 61 | 61 | 26 | 34 | 67 | 45 | 67 | 79 | 56 |
| Setup cost $s_t$ | 85 | 102 | 102 | 101 | 98 | 114 | 105 | 86 | 119 | 110 | 98 | 114 |
| Holding cost $h_t$ | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 0 |

## Reproducing the Wagner-Whitin Algorithm

Theorem 2 in the paper shows that an optimal order in month $j$ covers a consecutive block of demands through some month $t$. The cost of that final order is

$$C(j,t)=s_j+\sum_{k=j+1}^{t}d_k\sum_{h=j}^{k-1}h_h.$$

Let $F(t)$ be the minimum cost through month $t$, with $F(0)=0$. The forward algorithm evaluates

$$F(t)=\min_{1\le j\le t}\{F(j-1)+C(j,t)\}.$$

Saving the minimizing month $j$ at each step allows the final ordering schedule to be recovered by backtracking. The Planning Horizon Theorem permits later searches to begin at the previous minimizing order month.

In [ ]:
DEMAND = [69, 29, 36, 61, 61, 26, 34, 67, 45, 67, 79, 56]
SETUP_COST = [85, 102, 102, 101, 98, 114, 105, 86, 119, 110, 98, 114]
HOLDING_COST = [1] * 11 + [0]


def order_cost(order_period, last_period):
    holding = 0
    for demand_period in range(order_period + 1, last_period + 1):
        holding += DEMAND[demand_period] * sum(HOLDING_COST[order_period:demand_period])
    return SETUP_COST[order_period] + holding


def wagner_whitin():
    horizon = len(DEMAND)
    minimum_cost = [0] + [float("inf")] * horizon
    predecessor = [-1] * horizon
    earliest_candidate = 0

    for last_period in range(horizon):
        candidates = [
            (minimum_cost[order_period] + order_cost(order_period, last_period), order_period)
            for order_period in range(earliest_candidate, last_period + 1)
        ]
        minimum_cost[last_period + 1], predecessor[last_period] = min(candidates)
        earliest_candidate = predecessor[last_period]

    intervals = []
    last_period = horizon - 1
    while last_period >= 0:
        order_period = predecessor[last_period]
        quantity = sum(DEMAND[order_period : last_period + 1])
        intervals.append((order_period, last_period, quantity))
        last_period = order_period - 1

    orders = [0] * horizon
    for order_period, _, quantity in intervals:
        orders[order_period] = quantity

    inventory = []
    on_hand = 0
    for period in range(horizon):
        on_hand += orders[period] - DEMAND[period]
        inventory.append(on_hand)

    return minimum_cost[1:], list(reversed(intervals)), orders, inventory


minimum_cost, intervals, orders, inventory = wagner_whitin()

print("Month  Order  Ending inventory")
for month, (order, ending) in enumerate(zip(orders, inventory), start=1):
    print(f"{month:>5}  {order:>5}  {ending:>16}")

setup_total = sum(cost for cost, order in zip(SETUP_COST, orders) if order > 0)
holding_total = sum(cost * level for cost, level in zip(HOLDING_COST, inventory))
print(f"\nOrder months: {[period + 1 for period, _, _ in intervals]}")
print(f"Setup cost: {setup_total}")
print(f"Holding cost: {holding_total}")
print(f"Minimum total cost: {minimum_cost[-1]}")

assert minimum_cost == [85, 114, 186, 277, 348, 400, 469, 555, 600, 710, 789, 864]
assert orders == [98, 0, 97, 0, 121, 0, 0, 112, 0, 67, 135, 0]
assert setup_total == 579 and holding_total == 285 and minimum_cost[-1] == 864

The forward algorithm reproduces the paper's minimum cost of **864**. It orders 98 units in month 1, 97 in month 3, 121 in month 5, 112 in month 8, 67 in month 10, and 135 in month 11. The resulting setup cost is 579 and inventory-holding cost is 285.

## Exact Dynamic Lot-Sizing Formulation in OPL

The OPL model uses nonnegative order quantities $x_t$, nonnegative ending inventories $I_t$, and binary setup variables $y_t$. Inventory balance requires

$$I_{t-1}+x_t-I_t=d_t,$$

while $x_t\le My_t$ activates the setup cost whenever an order is placed. With $M=\sum_t d_t$, zero initial inventory, and $I_N=0$, this is an exact mixed-integer formulation of the same uncapacitated lot-sizing problem.

In [ ]:
model_text = '''
/**

Wagner-Whitin Dynamic Lot Sizing Problem

This model determines when and how much to order over a finite planning horizon
to satisfy deterministic, time-varying demand while minimizing fixed setup costs
and inventory holding costs. Inventory may be carried between periods, initial
inventory is implicitly zero, and backlogging is prohibited. A binary setup
variable activates each order through an aggregate-demand big-M bound.

*/

// § Index sets
// Number of planning periods, supplied by the data file.
int N = ...;

// Planning periods are numbered consecutively from 1 through N.
range Periods = 1..N;

// § Parameters
// demand[t]: demand that must be satisfied during period t.
param float demand[Periods];

// setupCost[t]: fixed cost incurred if an order is placed in period t.
param float setupCost[Periods];

// holdingCost[t]: cost per unit of inventory remaining after period t.
param float holdingCost[Periods];

// Derived aggregate demand used as a valid big-M bound on any single order.
param float totalDemand = sum(t in Periods) demand[t];

// § Decision variables
// orderQuantity[t]: nonnegative quantity ordered in period t.
dvar float+ orderQuantity[Periods];

// endingInventory[t]: nonnegative inventory remaining after period t;
// nonnegativity means that backlogging is not allowed.
dvar float+ endingInventory[Periods];

// orderPlaced[t]: 1 if an order is activated in period t, and 0 otherwise.
dvar boolean orderPlaced[Periods];

// § Objective
// Objective: minimize fixed ordering costs plus end-of-period holding costs.
minimize TotalCost:
  sum(t in Periods) (setupCost[t] * orderPlaced[t] + holdingCost[t] * endingInventory[t]);

subject to {
  // § Inventory-balance constraints
  // Constraint: later-period inventory flow balance.
  forall(t in 1..N)
    inventoryBalance:
      ((t > 1) ? endingInventory[t - 1] : 0) + orderQuantity[t] - endingInventory[t] == demand[t];

  // § Setup-activation constraints
  // Constraint: an order is allowed only when its setup is active.
  // totalDemand is the derived big-M upper bound.
  forall(t in Periods)
    setupActivation:
      orderQuantity[t] <= totalDemand * orderPlaced[t];
}
'''

data_text = '''
// § Planning horizon
// Twelve monthly planning periods.
N = 12;

// § Period data
// Demand in periods 1 through 12.
demand = [69, 29, 36, 61, 61, 26, 34, 67, 45, 67, 79, 56];

// Fixed ordering/setup cost in each period.
setupCost = [85, 102, 102, 101, 98, 114, 105, 86, 119, 110, 98, 114];

// Per-unit cost of inventory held after each period.
// The final-period cost is zero because terminal inventory is constrained to zero.
holdingCost = [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0];
'''

## Solving with PyOPL and HiGHS

PyOPL compiles the OPL model and delegates the mixed-integer program to HiGHS.

In [ ]:
%%capture
! pip install rhetor

In [ ]:
from pyopl import solve

model_file = "wagner_whitin.mod"
data_file = "wagner_whitin.dat"

with open(model_file, "w") as file:
    file.write(model_text)

with open(data_file, "w") as file:
    file.write(data_text)

results = solve(model_file, data_file, solver="scipy")
stats = results.get("stats", {})

print(f"Status: {results['status']}")
print(f"Objective: {results['objective_value']:.0f} cost units")
print(f"MIP gap: {stats.get('MIPGap', float('nan')):.2%}")
print(f"Runtime: {stats.get('Runtime', float('nan')):.2f} seconds")

assert results["status"] == "OPTIMAL"
assert abs(results["objective_value"] - 864) < 1e-6

## Extract and Validate the Ordering Plan

In [ ]:
import re

variable_pattern = re.compile(r"^(orderQuantity|endingInventory|orderPlaced)(?:\[(\d+)\]|_(\d+))$")


def extract_values(variable):
    values = [0.0] * 12
    for name, value in results["solution"].items():
        match = variable_pattern.fullmatch(name)
        if match and match.group(1) == variable:
            period = int(match.group(2) or match.group(3))
            values[period - 1] = value
    return values


pyopl_orders = extract_values("orderQuantity")
pyopl_inventory = extract_values("endingInventory")
pyopl_setups = extract_values("orderPlaced")

print("Month  Order  Ending inventory")
for month, (order, ending) in enumerate(zip(pyopl_orders, pyopl_inventory), start=1):
    print(f"{month:>5}  {order:>5.0f}  {ending:>16.0f}")

previous_inventory = 0
for period in range(12):
    assert abs(previous_inventory + pyopl_orders[period] - DEMAND[period] - pyopl_inventory[period]) < 1e-6
    previous_inventory = pyopl_inventory[period]

assert [round(n) for n in pyopl_orders] == orders
assert previous_inventory == 0
assert abs(sum(cost * value for cost, value in zip(SETUP_COST, pyopl_setups)) - 579) < 1e-6
assert abs(sum(cost * value for cost, value in zip(HOLDING_COST, pyopl_inventory)) - 285) < 1e-6
print(f"\nValidated minimum total cost: {results['objective_value']:.0f}")

## Comparison with the Wagner-Whitin Paper

| Method | Minimum cost | Interpretation |
|---|---:|---|
| Paper's forward dynamic program | 864 | Published optimal policy for the twelve-month example |
| Python reproduction | 864 | Direct implementation of the paper's recurrence and backtracking |
| OPL lot-sizing model solved by HiGHS | 864 | Independent mixed-integer optimum |

Both implementations order in months 1, 3, 5, 8, 10, and 11, with quantities 98, 97, 121, 112, 67, and 135.

## References

- H. M. Wagner and T. M. Whitin, "Dynamic Version of the Economic Lot Size Model," *Management Science*, 5(1), 89-96, 1958.
- H. M. Wagner and T. M. Whitin, "Dynamic Version of the Economic Lot Size Model," *Management Science*, 50(12), 1770-1774, 2004 reprint. [https://www.jstor.org/stable/30046142](https://www.jstor.org/stable/30046142)